# Compile Season Data

Get tournament matchup matrix for specified season

In [1]:
season = 2025

playin_losers = (  # remove play-in losers from seeding data
    3343,  # Princeton
    3380,  # Southern Univ
    3456,  # William & Mary
    3162,  # Columbia
)

model_path = '../data/models/womens/2025_03_15_model.pkl'
data_path = '../data/models/womens/2025_03_15_data.parquet'

season

2025

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\womens_kaggle\tournament_results.parquet')

df = df.loc[df['Season'] == season, :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2025,3101,Abilene Chr,-1.0,-1.00
1,2025,3102,Air Force,-1.0,-1.00
2,2025,3103,Akron,-1.0,-1.00
3,2025,3104,Alabama,1.0,0.25
4,2025,3105,Alabama A&M,-1.0,-1.00
...,...,...,...,...,...
373,2025,3476,Stonehill,-1.0,-1.00
374,2025,3477,East Texas A&M,-1.0,-1.00
375,2025,3478,Le Moyne,-1.0,-1.00
376,2025,3479,Mercyhurst,-1.0,-1.00


### Barttorvik Ratings

Omitted

In [3]:
# df_barttorvik = pd.read_parquet(r'..\data\preprocessed\womens_barttorvik\barttorvik.parquet')

# df_barttorvik = df_barttorvik.loc[df_barttorvik['Season'] == season, :].reset_index(drop=True)

# df_barttorvik

In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\WTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 3192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,3394
1,a&m-corpus christi,3394
2,abilene chr,3101
3,abilene christian,3101
4,abilene-christian,3101
...,...,...
1171,youngstown st.,3464
1172,youngstown state,3464
1173,youngstown-st,3464
1174,youngstown-state,3464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1170

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_21320\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

# df_match.head(25)

In [8]:
# df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

# df_barttorvik

In [9]:
# df = pd.merge(
#     df,
#     df_barttorvik.drop(columns=['TEAM']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

### Past Seasons

In [10]:
df_ps = pd.read_parquet(r'..\data\preprocessed\womens_past_seasons\past_seasons_ratings.parquet')

df_ps = df_ps.loc[df_ps['Season'] == season, :].reset_index(drop=True)

df_ps

,Season,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2025,Abilene Christian,-0.030986,-0.049325
1,2025,Air Force,-0.047323,-0.018258
2,2025,Akron,-0.108354,-0.047807
3,2025,Alabama,0.267055,0.233554
4,2025,Alabama A&M,-0.131853,-0.117158
...,...,...,...,...
358,2025,Wright State,-0.047642,-0.068898
359,2025,Wyoming,0.071691,0.079227
360,2025,Xavier,-0.205147,-0.105021
361,2025,Yale,-0.099924,-0.042498


In [11]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ps['Team'].unique())

df_match.head(25)

  0%|          | 0/363 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Hartford Hawks,hartford,73
1,St. Francis (NY) Terriers,st francis (ny),74
2,Savannah State Tigers,savannah state,80
3,Abilene Christian,abilene christian,100
4,Quinnipiac,quinnipiac,100
5,Queens (NC),queens (nc),100
6,Purdue Fort Wayne,purdue fort wayne,100
7,Purdue,purdue,100
8,Providence,providence,100
9,Princeton,princeton,100


In [12]:
df_ps.insert(1, 'TeamID', df_ps['Team'].map(team_to_spelling).map(spelling_to_id))

df_ps

,Season,TeamID,Team,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2025,3101,Abilene Christian,-0.030986,-0.049325
1,2025,3102,Air Force,-0.047323,-0.018258
2,2025,3103,Akron,-0.108354,-0.047807
3,2025,3104,Alabama,0.267055,0.233554
4,2025,3105,Alabama A&M,-0.131853,-0.117158
...,...,...,...,...,...
358,2025,3460,Wright State,-0.047642,-0.068898
359,2025,3461,Wyoming,0.071691,0.079227
360,2025,3462,Xavier,-0.205147,-0.105021
361,2025,3463,Yale,-0.099924,-0.042498


In [13]:
df = pd.merge(
    df,
    df_ps.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
0,2025,3101,Abilene Chr,-1.0,-1.00,-0.030986,-0.049325
1,2025,3102,Air Force,-1.0,-1.00,-0.047323,-0.018258
2,2025,3103,Akron,-1.0,-1.00,-0.108354,-0.047807
3,2025,3104,Alabama,1.0,0.25,0.267055,0.233554
4,2025,3105,Alabama A&M,-1.0,-1.00,-0.131853,-0.117158
...,...,...,...,...,...,...,...
373,2025,3476,Stonehill,-1.0,-1.00,-0.365483,-0.300154
374,2025,3477,East Texas A&M,-1.0,-1.00,-0.136236,-0.126214
375,2025,3478,Le Moyne,-1.0,-1.00,-0.127038,NaN
376,2025,3479,Mercyhurst,-1.0,-1.00,NaN,NaN


In [14]:
df.loc[df['Past Year Efficiency Margin'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin
8,2025,3109,Alliant Intl,-1.0,-1.0,NaN,NaN
17,2025,3118,Armstrong St,-1.0,-1.0,NaN,NaN
20,2025,3121,Augusta,-1.0,-1.0,NaN,NaN
27,2025,3128,Birmingham So,-1.0,-1.0,NaN,NaN
33,2025,3134,Brooklyn,-1.0,-1.0,NaN,NaN
46,2025,3147,Centenary,-1.0,-1.0,NaN,NaN
113,2025,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN
114,2025,3216,Hartford,-1.0,-1.0,NaN,-0.348182
187,2025,3289,Morris Brown,-1.0,-1.0,NaN,NaN
200,2025,3302,NE Illinois,-1.0,-1.0,NaN,NaN


In [15]:
df.loc[df['Past Year Efficiency Margin'].isna() & (df['Past 4 Years Tournament Results'] > -1.0), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin


### My Rankings

In [16]:
df_rankings = pd.read_parquet(fr'..\data\preprocessed\womens_my_rankings\my_rankings_{season}.parquet')

df_rankings.insert(0, 'Season', season)

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2025,South Carolina,5.147655,0.562154,1.205098,0.642944,72.154811
1,2025,UCLA,4.908453,0.481295,1.168855,0.687560,71.129603
2,2025,Texas,4.802689,0.522286,1.179051,0.656765,70.383361
3,2025,Connecticut,4.473622,0.583006,1.240840,0.657834,70.633882
4,2025,Southern California,4.409562,0.468648,1.140761,0.672113,75.836752
...,...,...,...,...,...,...,...
357,2025,Long Island University,-3.460222,-0.376161,0.702026,1.078187,68.350150
358,2025,Arkansas-Pine Bluff,-3.558144,-0.316465,0.694982,1.011448,72.058559
359,2025,UNC Asheville,-3.661215,-0.325492,0.727043,1.052535,71.835725
360,2025,South Carolina State,-3.805312,-0.404193,0.681998,1.086191,69.509904


In [17]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,Cornell,cornell,100
2,Marist,marist,100
3,Northern Illinois,northern illinois,100
4,Elon,elon,100
5,George Washington,george washington,100
6,Valparaiso,valparaiso,100
7,Austin Peay,austin peay,100
8,Delaware,delaware,100
9,California Baptist,california baptist,100


In [18]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2025,3376,South Carolina,5.147655,0.562154,1.205098,0.642944,72.154811
1,2025,3417,UCLA,4.908453,0.481295,1.168855,0.687560,71.129603
2,2025,3400,Texas,4.802689,0.522286,1.179051,0.656765,70.383361
3,2025,3163,Connecticut,4.473622,0.583006,1.240840,0.657834,70.633882
4,2025,3425,Southern California,4.409562,0.468648,1.140761,0.672113,75.836752
...,...,...,...,...,...,...,...,...
357,2025,3254,Long Island University,-3.460222,-0.376161,0.702026,1.078187,68.350150
358,2025,3115,Arkansas-Pine Bluff,-3.558144,-0.316465,0.694982,1.011448,72.058559
359,2025,3421,UNC Asheville,-3.661215,-0.325492,0.727043,1.052535,71.835725
360,2025,3354,South Carolina State,-3.805312,-0.404193,0.681998,1.086191,69.509904


In [19]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [20]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2025,3101,Abilene Chr,-1.0,-1.00,-0.030986,-0.049325,-0.348582,-0.003934,0.926011,0.929945,70.532946
1,2025,3102,Air Force,-1.0,-1.00,-0.047323,-0.018258,-0.241765,-0.008882,0.919724,0.928606,70.083276
2,2025,3103,Akron,-1.0,-1.00,-0.108354,-0.047807,-1.228059,-0.156519,0.853280,1.009799,69.820430
3,2025,3104,Alabama,1.0,0.25,0.267055,0.233554,2.983266,0.368523,1.120087,0.751564,71.747444
4,2025,3105,Alabama A&M,-1.0,-1.00,-0.131853,-0.117158,-0.621060,-0.075576,0.861589,0.937165,70.415144
...,...,...,...,...,...,...,...,...,...,...,...,...
373,2025,3476,Stonehill,-1.0,-1.00,-0.365483,-0.300154,-2.295418,-0.217392,0.838302,1.055694,70.290705
374,2025,3477,East Texas A&M,-1.0,-1.00,-0.136236,-0.126214,-2.090635,-0.169735,0.836489,1.006224,71.555109
375,2025,3478,Le Moyne,-1.0,-1.00,-0.127038,NaN,-2.981242,-0.338639,0.750252,1.088890,67.506824
376,2025,3479,Mercyhurst,-1.0,-1.00,NaN,NaN,-3.002792,-0.269764,0.781476,1.051240,73.362336


In [21]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
8,2025,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2025,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2025,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2025,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2025,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2025,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2025,3216,Hartford,-1.0,-1.0,NaN,-0.348182,NaN,NaN,NaN,NaN,NaN
187,2025,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2025,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Starters

Omitted

In [22]:
# df_starters = pd.read_parquet(fr'..\data\preprocessed\womens_starters\starters_{season}.parquet')

# df_starters.insert(0, 'Season', season)

# df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

# df_starters

In [23]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

# df_match.head(25)

In [24]:
# df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

# df_starters

In [25]:
# df_starters.loc[df_starters['TeamID'].isna(), :]

In [26]:
# df = pd.merge(
#     df,
#     df_starters.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [27]:
# df.loc[df['Starters'].isna(), :]

### Openskill Ratings

In [28]:
df_os = pd.read_parquet(fr'..\data\preprocessed\womens_os_rankings\os_rankings_{season}.parquet')

df_os.insert(0, 'Season', season)

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2025,South Carolina,57.154437,44.931333
1,2025,UCLA,56.918023,44.291517
2,2025,Texas,55.779370,43.379118
3,2025,Southern California,55.653531,42.960902
4,2025,Connecticut,54.559248,42.003074
...,...,...,...,...
357,2025,Cal State Bakersfield,0.257786,-14.094578
358,2025,American,0.357573,-14.645875
359,2025,Delaware State,-0.959417,-15.475132
360,2025,South Carolina State,-1.864294,-15.979296


In [29]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,South Carolina,south carolina,100
1,Northern Illinois,northern illinois,100
2,Hofstra,hofstra,100
3,Boston University,boston university,100
4,Santa Clara,santa clara,100
5,Northern Colorado,northern colorado,100
6,Sacramento State,sacramento state,100
7,West Georgia,west georgia,100
8,NJIT,njit,100
9,Western Michigan,western michigan,100


In [30]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2025,3376,South Carolina,57.154437,44.931333
1,2025,3417,UCLA,56.918023,44.291517
2,2025,3400,Texas,55.779370,43.379118
3,2025,3425,Southern California,55.653531,42.960902
4,2025,3163,Connecticut,54.559248,42.003074
...,...,...,...,...,...
357,2025,3167,Cal State Bakersfield,0.257786,-14.094578
358,2025,3110,American,0.357573,-14.645875
359,2025,3175,Delaware State,-0.959417,-15.475132
360,2025,3354,South Carolina State,-1.864294,-15.979296


In [31]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating
0,2025,3101,Abilene Chr,-1.0,-1.00,-0.030986,-0.049325,-0.348582,-0.003934,0.926011,0.929945,70.532946,26.161976,12.912197
1,2025,3102,Air Force,-1.0,-1.00,-0.047323,-0.018258,-0.241765,-0.008882,0.919724,0.928606,70.083276,25.521376,12.701676
2,2025,3103,Akron,-1.0,-1.00,-0.108354,-0.047807,-1.228059,-0.156519,0.853280,1.009799,69.820430,10.844494,-2.119788
3,2025,3104,Alabama,1.0,0.25,0.267055,0.233554,2.983266,0.368523,1.120087,0.751564,71.747444,43.257514,30.568372
4,2025,3105,Alabama A&M,-1.0,-1.00,-0.131853,-0.117158,-0.621060,-0.075576,0.861589,0.937165,70.415144,23.417125,9.739409
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2025,3476,Stonehill,-1.0,-1.00,-0.365483,-0.300154,-2.295418,-0.217392,0.838302,1.055694,70.290705,17.580177,5.059065
374,2025,3477,East Texas A&M,-1.0,-1.00,-0.136236,-0.126214,-2.090635,-0.169735,0.836489,1.006224,71.555109,8.619987,-5.707010
375,2025,3478,Le Moyne,-1.0,-1.00,-0.127038,NaN,-2.981242,-0.338639,0.750252,1.088890,67.506824,6.809705,-6.077318
376,2025,3479,Mercyhurst,-1.0,-1.00,NaN,NaN,-3.002792,-0.269764,0.781476,1.051240,73.362336,5.548606,-7.932131


In [32]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating
8,2025,3109,Alliant Intl,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2025,3118,Armstrong St,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2025,3121,Augusta,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025,3128,Birmingham So,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2025,3134,Brooklyn,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2025,3147,Centenary,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
113,2025,3215,Hardin-Simmons,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2025,3216,Hartford,-1.0,-1.0,NaN,-0.348182,NaN,NaN,NaN,NaN,NaN,NaN,NaN
187,2025,3289,Morris Brown,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
200,2025,3302,NE Illinois,-1.0,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Standard Stats

Omitted

In [33]:
# df_ss = pd.read_parquet('../data/preprocessed/womens_standard_stats/standard_stats.parquet')

# df_ss = df_ss.loc[df_ss['Season'] == season, :].reset_index(drop=True)

# df_ss

In [34]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_ss['Team'].unique())

# df_match.head(25)

In [35]:
# df_ss.insert(1, 'TeamID', df_ss['Team'].map(team_to_spelling).map(spelling_to_id))

# df_ss

In [36]:
# df = pd.merge(
#     df,
#     df_ss.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [37]:
# df.loc[df['Team Win%'].isna(), :]

### Map to Matchups

In [38]:
# df_seeds = pd.read_csv(fr'..\data\unprocessed\kaggle\{season}_tourney_seeds.csv')

# df_seeds = df_seeds.loc[df_seeds['Tournament'] == 'M', :].reset_index(drop=True)

# df_seeds.rename(columns={'Seed': 'Region Seed'}, inplace=True)
# df_seeds.insert(2, 'Region', df_seeds['Region Seed'].str[0])
# df_seeds.insert(3, 'Seed', df_seeds['Region Seed'].str.extract('(\d+)').astype(int))

# df_seeds

In [39]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\WNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

# df_seeds = df_seeds.loc[~df_seeds['TeamID'].isin(playin_losers), :].reset_index(drop=True)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2025,1,W,False,3376
1,2025,2,W,False,3181
2,2025,3,W,False,3314
3,2025,4,W,False,3268
4,2025,5,W,False,3104
...,...,...,...,...,...
63,2025,12,Z,False,3193
64,2025,13,Z,False,3251
65,2025,14,Z,False,3195
66,2025,15,Z,False,3117


In [40]:
df = df.merge(
    df_seeds,
    how='left',
    on=['Season', 'TeamID'],
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Seed,Region,Play In
0,2025,3101,Abilene Chr,-1.0,-1.00,-0.030986,-0.049325,-0.348582,-0.003934,0.926011,0.929945,70.532946,26.161976,12.912197,NaN,NaN,NaN
1,2025,3102,Air Force,-1.0,-1.00,-0.047323,-0.018258,-0.241765,-0.008882,0.919724,0.928606,70.083276,25.521376,12.701676,NaN,NaN,NaN
2,2025,3103,Akron,-1.0,-1.00,-0.108354,-0.047807,-1.228059,-0.156519,0.853280,1.009799,69.820430,10.844494,-2.119788,NaN,NaN,NaN
3,2025,3104,Alabama,1.0,0.25,0.267055,0.233554,2.983266,0.368523,1.120087,0.751564,71.747444,43.257514,30.568372,5.0,W,False
4,2025,3105,Alabama A&M,-1.0,-1.00,-0.131853,-0.117158,-0.621060,-0.075576,0.861589,0.937165,70.415144,23.417125,9.739409,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,2025,3476,Stonehill,-1.0,-1.00,-0.365483,-0.300154,-2.295418,-0.217392,0.838302,1.055694,70.290705,17.580177,5.059065,NaN,NaN,NaN
374,2025,3477,East Texas A&M,-1.0,-1.00,-0.136236,-0.126214,-2.090635,-0.169735,0.836489,1.006224,71.555109,8.619987,-5.707010,NaN,NaN,NaN
375,2025,3478,Le Moyne,-1.0,-1.00,-0.127038,NaN,-2.981242,-0.338639,0.750252,1.088890,67.506824,6.809705,-6.077318,NaN,NaN,NaN
376,2025,3479,Mercyhurst,-1.0,-1.00,NaN,NaN,-3.002792,-0.269764,0.781476,1.051240,73.362336,5.548606,-7.932131,NaN,NaN,NaN


Remap to Team A / Team B format

In [41]:
id_to_region = dict(zip(df['TeamID'], df['Region']))
id_to_seed = dict(zip(df['TeamID'], df['Seed']))

df_mod = pd.DataFrame(
    [
        (team_a, team_b) 
        for team_a in df['TeamID'].unique() 
        for team_b in df['TeamID'].unique() 
        if team_a != team_b
    ],
    columns=['Team A ID', 'Team B ID']
)

df_mod.insert(0, 'Season', season)
df_mod['Team A Region'] = df_mod['Team A ID'].map(id_to_region)
df_mod['Team B Region'] = df_mod['Team B ID'].map(id_to_region)
df_mod['Team A Seed'] = df_mod['Team A ID'].map(id_to_seed)
df_mod['Team B Seed'] = df_mod['Team B ID'].map(id_to_seed)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2025,3101,3102,NaN,NaN,NaN,NaN
1,2025,3101,3103,NaN,NaN,NaN,NaN
2,2025,3101,3104,NaN,W,NaN,5.0
3,2025,3101,3105,NaN,NaN,NaN,NaN
4,2025,3101,3106,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
142501,2025,3480,3475,NaN,NaN,NaN,NaN
142502,2025,3480,3476,NaN,NaN,NaN,NaN
142503,2025,3480,3477,NaN,NaN,NaN,NaN
142504,2025,3480,3478,NaN,NaN,NaN,NaN


Get round of matchup

In [42]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # ignore play-in games

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0         False
1         False
2         False
3         False
4         False
          ...  
142501    False
142502    False
142503    False
142504    False
142505    False
Length: 142506, dtype: bool

In [43]:
df_mod['Round'] = float('nan')

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod['Round'].describe()

count    4548.000000
mean        5.095866
std         1.190665
min         1.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         6.000000
Name: Round, dtype: float64

Get home court advantage

In [44]:
df_mod['Location'] = 0

# conditions: after 2012 but not 2021 (covid stadium), matchup is within first 2 rounds, and team is top 4 seed
df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team A Seed'] <= 4), 
    'Location'
] = 1

df_mod.loc[
    (df_mod['Season'] > 2012) & 
    (df_mod['Season'] != 2021) & 
    (df_mod['Round'] <= 2) & 
    (df_mod['Team B Seed'] <= 4), 
    'Location'
] = -1

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location
0,2025,3101,3102,NaN,NaN,NaN,NaN,NaN,0
1,2025,3101,3103,NaN,NaN,NaN,NaN,NaN,0
2,2025,3101,3104,NaN,W,NaN,5.0,NaN,0
3,2025,3101,3105,NaN,NaN,NaN,NaN,NaN,0
4,2025,3101,3106,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...
142501,2025,3480,3475,NaN,NaN,NaN,NaN,NaN,0
142502,2025,3480,3476,NaN,NaN,NaN,NaN,NaN,0
142503,2025,3480,3477,NaN,NaN,NaN,NaN,NaN,0
142504,2025,3480,3478,NaN,NaN,NaN,NaN,NaN,0


In [45]:
df_mod['Seed'] = df_mod['Team A Seed'] - df_mod['Team B Seed']

# df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed
0,2025,3101,3102,NaN,NaN,NaN,NaN,NaN,0,NaN
1,2025,3101,3103,NaN,NaN,NaN,NaN,NaN,0,NaN
2,2025,3101,3104,NaN,W,NaN,5.0,NaN,0,NaN
3,2025,3101,3105,NaN,NaN,NaN,NaN,NaN,0,NaN
4,2025,3101,3106,NaN,NaN,NaN,NaN,NaN,0,NaN
...,...,...,...,...,...,...,...,...,...,...
142501,2025,3480,3475,NaN,NaN,NaN,NaN,NaN,0,NaN
142502,2025,3480,3476,NaN,NaN,NaN,NaN,NaN,0,NaN
142503,2025,3480,3477,NaN,NaN,NaN,NaN,NaN,0,NaN
142504,2025,3480,3478,NaN,NaN,NaN,NaN,NaN,0,NaN


Get Head-to-Head

In [46]:
df_h2h = pd.read_parquet('../data/preprocessed/womens_h2h/h2h.parquet')

df_h2h = df_h2h.loc[df_h2h['Season'] == season, :].reset_index(drop=True)

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2025,Abilene Christian,Air Force,NaN,0.120456
1,2025,Abilene Christian,Alabama A&M,NaN,0.342959
2,2025,Abilene Christian,Alcorn State,NaN,0.080103
3,2025,Abilene Christian,Arizona,NaN,-0.837085
4,2025,Abilene Christian,Arizona State,NaN,-0.906969
...,...,...,...,...,...
69901,2025,Youngstown State,Western Michigan,NaN,0.328056
69902,2025,Youngstown State,William & Mary,NaN,-1.439793
69903,2025,Youngstown State,Wisconsin,NaN,-0.871134
69904,2025,Youngstown State,Wright State,-0.777343,0.245841


In [47]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/362 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Abilene Christian,abilene christian,100
1,Queens (NC),queens (nc),100
2,Purdue Fort Wayne,purdue fort wayne,100
3,Purdue,purdue,100
4,Providence,providence,100
5,Princeton,princeton,100
6,Presbyterian,presbyterian,100
7,Prairie View,prairie view,100
8,Portland State,portland state,100
9,Quinnipiac,quinnipiac,100


In [48]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2025,3101,Abilene Christian,3102,Air Force,NaN,0.120456
1,2025,3101,Abilene Christian,3105,Alabama A&M,NaN,0.342959
2,2025,3101,Abilene Christian,3108,Alcorn State,NaN,0.080103
3,2025,3101,Abilene Christian,3112,Arizona,NaN,-0.837085
4,2025,3101,Abilene Christian,3113,Arizona State,NaN,-0.906969
...,...,...,...,...,...,...,...
69901,2025,3464,Youngstown State,3444,Western Michigan,NaN,0.328056
69902,2025,3464,Youngstown State,3456,William & Mary,NaN,-1.439793
69903,2025,3464,Youngstown State,3458,Wisconsin,NaN,-0.871134
69904,2025,3464,Youngstown State,3460,Wright State,-0.777343,0.245841


In [49]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps
0,2025,3101,3102,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.120456
1,2025,3101,3103,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
2,2025,3101,3104,NaN,W,NaN,5.0,NaN,0,NaN,NaN,NaN
3,2025,3101,3105,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.342959
4,2025,3101,3106,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2025,3480,3475,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142502,2025,3480,3476,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142503,2025,3480,3477,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142504,2025,3480,3478,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN


Get team names

In [50]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\WTeams.csv')

df_teams

,TeamID,TeamName
0,3101,Abilene Chr
1,3102,Air Force
2,3103,Akron
3,3104,Alabama
4,3105,Alabama A&M
...,...,...
373,3476,Stonehill
374,3477,East Texas A&M
375,3478,Le Moyne
376,3479,Mercyhurst


In [51]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps
0,2025,3101,Abilene Chr,3102,Air Force,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.120456
1,2025,3101,Abilene Chr,3103,Akron,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
2,2025,3101,Abilene Chr,3104,Alabama,NaN,W,NaN,5.0,NaN,0,NaN,NaN,NaN
3,2025,3101,Abilene Chr,3105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.342959
4,2025,3101,Abilene Chr,3106,Alabama St,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2025,3480,West Georgia,3475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142502,2025,3480,West Georgia,3476,Stonehill,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142503,2025,3480,West Georgia,3477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN
142504,2025,3480,West Georgia,3478,Le Moyne,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN


Map features

In [52]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

assert (df_mod.shape[0] == team_a_features.shape[0]), 'There is an issue with merging'
assert (df_mod.shape[0] == team_b_features.shape[0]), 'There is an issue with merging'

df_features = team_a_features - team_b_features

# df_features['Team A ADJOE Team B ADJDE'] = team_a_features['ADJOE'] + team_b_features['ADJDE']
# df_features['Team B ADJOE Team A ADJDE'] = team_b_features['ADJOE'] + team_a_features['ADJDE']

# df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
# df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A Efficiency Margin'] = team_a_features['Efficiency Margin']
df_features['Team B Efficiency Margin'] = team_b_features['Efficiency Margin']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,0.0,0.00,0.016338,-0.031067,-0.106816,0.004948,0.006287,0.001339,0.449669,0.640600,0.210521,-0.003934,-0.008882
1,0.0,0.00,0.077368,-0.001518,0.879477,0.152584,0.072731,-0.079853,0.712516,15.317483,15.031985,-0.003934,-0.156519
2,-2.0,-1.25,-0.298040,-0.282879,-3.331848,-0.372458,-0.194076,0.178382,-1.214499,-17.095537,-17.656175,-0.003934,0.368523
3,0.0,0.00,0.100868,0.067832,0.272479,0.071641,0.064422,-0.007220,0.117802,2.744852,3.172789,-0.003934,-0.075576
4,0.0,0.00,0.289596,0.110494,2.659595,0.334664,0.232805,-0.101860,1.407083,24.248409,24.215028,-0.003934,-0.338599
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,0.0,0.00,NaN,NaN,-1.109221,-0.148745,-0.090370,0.058375,1.028553,-5.986158,-6.207348,-0.136479,0.012266
142502,0.0,0.00,NaN,NaN,1.164371,0.080913,0.013384,-0.067529,1.816523,1.640451,1.474653,-0.136479,-0.217392
142503,0.0,0.00,NaN,NaN,0.959589,0.033256,0.015197,-0.018059,0.552119,10.600641,12.240728,-0.136479,-0.169735
142504,0.0,0.00,NaN,NaN,1.850195,0.202160,0.101435,-0.100725,4.600403,12.410923,12.611035,-0.136479,-0.338639


In [53]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,2025,3101,Abilene Chr,3102,Air Force,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.120456,0.0,0.00,0.016338,-0.031067,-0.106816,0.004948,0.006287,0.001339,0.449669,0.640600,0.210521,-0.003934,-0.008882
1,2025,3101,Abilene Chr,3103,Akron,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.00,0.077368,-0.001518,0.879477,0.152584,0.072731,-0.079853,0.712516,15.317483,15.031985,-0.003934,-0.156519
2,2025,3101,Abilene Chr,3104,Alabama,NaN,W,NaN,5.0,NaN,0,NaN,NaN,NaN,-2.0,-1.25,-0.298040,-0.282879,-3.331848,-0.372458,-0.194076,0.178382,-1.214499,-17.095537,-17.656175,-0.003934,0.368523
3,2025,3101,Abilene Chr,3105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,0.342959,0.0,0.00,0.100868,0.067832,0.272479,0.071641,0.064422,-0.007220,0.117802,2.744852,3.172789,-0.003934,-0.075576
4,2025,3101,Abilene Chr,3106,Alabama St,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.00,0.289596,0.110494,2.659595,0.334664,0.232805,-0.101860,1.407083,24.248409,24.215028,-0.003934,-0.338599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2025,3480,West Georgia,3475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,-1.109221,-0.148745,-0.090370,0.058375,1.028553,-5.986158,-6.207348,-0.136479,0.012266
142502,2025,3480,West Georgia,3476,Stonehill,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,1.164371,0.080913,0.013384,-0.067529,1.816523,1.640451,1.474653,-0.136479,-0.217392
142503,2025,3480,West Georgia,3477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,0.959589,0.033256,0.015197,-0.018059,0.552119,10.600641,12.240728,-0.136479,-0.169735
142504,2025,3480,West Georgia,3478,Le Moyne,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,1.850195,0.202160,0.101435,-0.100725,4.600403,12.410923,12.611035,-0.136479,-0.338639


In [54]:
# track if game is a tournament matchup for later
tournament_matchup = (df_mod['Team A Region'].notna()) & (df_mod['Team B Region'].notna())

tournament_matchup.sum()

4556

In [55]:
df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Round,Location,Seed,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Mu,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin
0,2025,3101,Abilene Chr,3102,Air Force,NaN,0,NaN,NaN,0.120456,0.0,0.00,0.016338,-0.031067,-0.106816,0.004948,0.006287,0.001339,0.449669,0.640600,0.210521,-0.003934,-0.008882
1,2025,3101,Abilene Chr,3103,Akron,NaN,0,NaN,NaN,NaN,0.0,0.00,0.077368,-0.001518,0.879477,0.152584,0.072731,-0.079853,0.712516,15.317483,15.031985,-0.003934,-0.156519
2,2025,3101,Abilene Chr,3104,Alabama,NaN,0,NaN,NaN,NaN,-2.0,-1.25,-0.298040,-0.282879,-3.331848,-0.372458,-0.194076,0.178382,-1.214499,-17.095537,-17.656175,-0.003934,0.368523
3,2025,3101,Abilene Chr,3105,Alabama A&M,NaN,0,NaN,NaN,0.342959,0.0,0.00,0.100868,0.067832,0.272479,0.071641,0.064422,-0.007220,0.117802,2.744852,3.172789,-0.003934,-0.075576
4,2025,3101,Abilene Chr,3106,Alabama St,NaN,0,NaN,NaN,NaN,0.0,0.00,0.289596,0.110494,2.659595,0.334664,0.232805,-0.101860,1.407083,24.248409,24.215028,-0.003934,-0.338599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
142501,2025,3480,West Georgia,3475,Southern Indiana,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,-1.109221,-0.148745,-0.090370,0.058375,1.028553,-5.986158,-6.207348,-0.136479,0.012266
142502,2025,3480,West Georgia,3476,Stonehill,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,1.164371,0.080913,0.013384,-0.067529,1.816523,1.640451,1.474653,-0.136479,-0.217392
142503,2025,3480,West Georgia,3477,East Texas A&M,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,0.959589,0.033256,0.015197,-0.018059,0.552119,10.600641,12.240728,-0.136479,-0.169735
142504,2025,3480,West Georgia,3478,Le Moyne,NaN,0,NaN,NaN,NaN,0.0,0.00,NaN,NaN,1.850195,0.202160,0.101435,-0.100725,4.600403,12.410923,12.611035,-0.136479,-0.338639


Drop features that were not used in the model

In [56]:
df_mod.drop(
    columns=[
        'Round',
        'Mu',
        'Adjusted Tempo',
        'Location',
        'Common Opps',
    ],
    inplace=True,
)

Check that data follows same format as the data that the model was trained on

In [57]:
df_mod_training = pd.read_parquet(data_path)

assert all(df_mod_training.drop(columns=['Result']).columns == df_mod.columns), 'Columns do not match'

'Columns Match'

'Columns Match'

### Get Model Predictions

In [58]:
import pickle

with open(model_path, 'rb') as f:
    mod = pickle.load(f)

mod

LGBMClassifier(early_stopping_round=25, feature_fraction=0.778518753537347,
               lambda_l1=2.096041209255663, lambda_l2=0.24121410842083418,
               learning_rate=0.025864767729102435, max_depth=8, metric='rmse',
               min_child_samples=3,
               monotone_constraints=[-1, 1, 1, 1, 1, 1, 1, 1, 1, -1, 1, 1, -1],
               n_estimators=1000, num_leaves=155, random_state=22,
               verbosity=-1)

In [59]:
X = df_mod.drop(columns=['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B'])

df_mod['Prediction'] = mod.predict_proba(X)[:, 1]

df_mod['Prediction']

0         0.658048
1         0.743218
2         0.059958
3         0.712673
4         0.930600
            ...   
142501    0.266477
142502    0.634497
142503    0.657146
142504    0.851471
142505    0.850796
Name: Prediction, Length: 142506, dtype: float64

In [60]:
(
    df_mod[['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B', 'Prediction']]
    .to_parquet(f'../data/simulations/womens/matchup_predictions_{season}.parquet')
)

'Done'

'Done'

Turn predictions into matchup matrix

In [61]:
# filter down to just tournament games
df_mod = df_mod.loc[tournament_matchup, :].reset_index(drop=True)

# filter out play-in losers
df_mod = df_mod.loc[
    (~df_mod['Team A ID'].isin(playin_losers)) & (~df_mod['Team B ID'].isin(playin_losers)), 
    :
].reset_index(drop=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Seed,Head to Head,Past Year Tournament Result,Past 4 Years Tournament Results,Past Year Efficiency Margin,Past 4 Years Efficiency Margin,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,OS Rating,Team A Efficiency Margin,Team B Efficiency Margin,Prediction
0,2025,3104,Alabama,3117,Arkansas St,-10.0,NaN,2.0,1.25,0.292891,0.311038,2.739477,0.325013,0.188056,-0.136957,10.296126,0.368523,0.043510,0.954425
1,2025,3104,Alabama,3123,Ball St,-7.0,NaN,2.0,1.25,0.127373,0.143133,1.041034,0.215022,0.134900,-0.080122,2.644335,0.368523,0.153502,0.862467
2,2025,3104,Alabama,3124,Baylor,1.0,NaN,-1.0,-1.50,-0.069174,-0.129966,-0.140351,-0.008115,0.015741,0.023856,-6.283273,0.368523,0.376638,0.372033
3,2025,3104,Alabama,3143,California,-3.0,-0.571735,2.0,1.25,0.096628,0.141799,0.457063,0.093856,0.047456,-0.046400,0.448978,0.368523,0.274668,0.541847
4,2025,3104,Alabama,3163,Connecticut,3.0,NaN,-3.0,-3.50,-0.213869,-0.229189,-1.490356,-0.214482,-0.120753,0.093729,-11.434702,0.368523,0.583006,0.087864
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2025,3471,UC San Diego,3435,Vanderbilt,9.0,NaN,-1.0,-0.25,-0.283016,-0.193279,-3.298378,-0.354995,-0.292839,0.062156,-11.118442,-0.008117,0.346878,0.046428
4028,2025,3471,UC San Diego,3436,Vermont,1.0,NaN,0.0,-0.25,-0.133911,-0.073827,-1.013357,-0.119960,-0.115563,0.004397,-5.849895,-0.008117,0.111843,0.272731
4029,2025,3471,UC San Diego,3449,Washington,5.0,NaN,0.0,0.00,-0.295857,-0.235186,-2.606726,-0.265014,-0.238249,0.026765,-10.483599,-0.008117,0.256896,0.061890
4030,2025,3471,UC San Diego,3452,West Virginia,10.0,NaN,-2.0,-1.25,-0.416016,-0.315787,-3.356618,-0.412555,-0.233642,0.178913,-17.707622,-0.008117,0.404437,0.046241


In [62]:
df_matrix = (
    df_mod[['Team A ID', 'Team B ID', 'Prediction']]
    .pivot(
        index=['Team A ID'], 
        columns=['Team B ID'],
        values='Prediction',
    )
)

df_matrix

Team B ID,3104,3117,3123,3124,3143,3163,3166,3181,3192,3193,3195,3199,3206,3210,3213,3217,3219,3228,3231,3234,3235,3243,3246,3250,3251,3257,3261,3268,3276,3277,3279,3280,3286,3293,3301,3304,3313,3314,3323,3326,3328,3329,3332,3333,3350,3355,3361,3372,3376,3378,3395,3397,3399,3400,3417,3422,3425,3428,3435,3436,3449,3452,3453,3471
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3104,NaN,0.954425,0.862467,0.372033,0.541847,0.087864,0.652267,0.342667,0.954425,0.835100,0.884582,0.640815,0.874335,0.753463,0.750500,0.768107,0.954425,0.764072,0.581044,0.550366,0.570316,0.287321,0.455343,0.952046,0.928469,0.641058,0.443561,0.497065,0.734790,0.829342,0.600898,0.856105,0.875358,0.921048,0.409789,0.684127,0.950864,0.724483,0.247309,0.595001,0.252251,0.739176,0.766506,0.942274,0.845440,0.744695,0.952046,0.947817,0.077059,0.921624,0.416816,0.185871,0.952046,0.081202,0.140245,0.952046,0.139300,0.742769,0.374175,0.952477,0.758456,0.367937,0.921361,0.954425
3117,0.046241,NaN,0.136014,0.046241,0.074816,0.046241,0.061890,0.046241,0.676768,0.138150,0.123514,0.048721,0.141297,0.067108,0.138150,0.080245,0.717240,0.066815,0.079564,0.046241,0.066054,0.046241,0.046672,0.301205,0.151207,0.065794,0.046241,0.046428,0.046241,0.061645,0.046241,0.066586,0.256361,0.259487,0.046241,0.080969,0.159623,0.046241,0.046241,0.046241,0.046241,0.061645,0.082139,0.389568,0.075024,0.118245,0.292961,0.263546,0.046241,0.136279,0.046241,0.048721,0.341281,0.046241,0.046241,0.476526,0.046241,0.065671,0.065774,0.292612,0.080969,0.046241,0.132868,0.714069
3123,0.140053,0.859297,NaN,0.127437,0.245238,0.046241,0.236148,0.046241,0.862661,0.413658,0.305752,0.244499,0.363310,0.264219,0.339524,0.270448,0.875467,0.263888,0.220087,0.175675,0.236148,0.129318,0.242248,0.825866,0.559752,0.221183,0.048721,0.203733,0.224457,0.221495,0.117810,0.264219,0.319590,0.316750,0.086836,0.261089,0.669545,0.105485,0.048721,0.130390,0.056356,0.245238,0.274833,0.654809,0.244614,0.141442,0.768335,0.586236,0.046241,0.448056,0.052144,0.136411,0.781152,0.046241,0.046241,0.859962,0.049795,0.257495,0.263305,0.823036,0.263761,0.117262,0.336277,0.874217
3124,0.620468,0.954425,0.878050,NaN,0.785477,0.142686,0.653449,0.364951,0.954425,0.876843,0.901825,0.706472,0.876109,0.742643,0.841218,0.789013,0.954425,0.803046,0.530797,0.735331,0.801006,0.643303,0.742140,0.952477,0.952477,0.763242,0.323683,0.647489,0.685385,0.750671,0.449941,0.797942,0.910697,0.841170,0.550215,0.691038,0.952477,0.749827,0.249880,0.704612,0.490489,0.607890,0.600344,0.950395,0.783243,0.766673,0.954425,0.954425,0.125606,0.934422,0.327459,0.373152,0.954425,0.142249,0.084966,0.954425,0.216817,0.798224,0.671117,0.954425,0.761857,0.703876,0.928378,0.954425
3143,0.392624,0.931992,0.747924,0.209767,NaN,0.059661,0.277877,0.130213,0.939002,0.738938,0.552610,0.499918,0.721801,0.526839,0.731626,0.704961,0.939002,0.315131,0.369112,0.225528,0.326007,0.253563,0.252194,0.861248,0.748786,0.174561,0.206469,0.241574,0.232640,0.153332,0.211973,0.315919,0.750656,0.646526,0.293978,0.444458,0.751396,0.169515,0.075503,0.218345,0.211883,0.371675,0.535098,0.895003,0.516256,0.476825,0.858663,0.855659,0.059661,0.654836,0.122023,0.249753,0.919149,0.059661,0.108926,0.925169,0.116708,0.265431,0.264312,0.859605,0.452633,0.209198,0.728901,0.939002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3436,0.048721,0.703111,0.175389,0.046241,0.135181,0.046241,0.113782,0.046241,0.720350,0.160107,0.197958,0.056581,0.200513,0.126121,0.169358,0.146780,0.725169,0.136932,0.122464,0.053509,0.123311,0.048721,0.047002,0.674722,0.299534,0.121718,0.046241,0.082585,0.053509,0.065671,0.048721,0.123463,0.309202,0.260307,0.046241,0.128394,0.370151,0.053509,0.046241,0.046241,0.046241,0.113282,0.127838,0.522304,0.126667,0.118245,0.668074,0.313668,0.046241,0.259239,0

In [63]:
df_matrix_display = df_matrix.copy()

df_matrix_display.columns = df_matrix_display.columns.map(id_to_team)
df_matrix_display.index = df_matrix_display.index.map(id_to_team)

df_matrix_display

Team B ID,Alabama,Arkansas St,Ball St,Baylor,California,Connecticut,Creighton,Duke,F Dickinson,Fairfield,FGCU,Florida St,George Mason,Georgia Tech,Grand Canyon,Harvard,High Point,Illinois,Indiana,Iowa,Iowa St,Kansas St,Kentucky,Lehigh,Liberty,Louisville,LSU,Maryland,Michigan,Michigan St,Mississippi,Mississippi St,Montana St,Murray St,NC State,Nebraska,Norfolk St,North Carolina,Notre Dame,Ohio St,Oklahoma,Oklahoma St,Oregon,Oregon St,Richmond,S Dakota St,San Diego St,SF Austin,South Carolina,South Florida,TCU,Tennessee,Tennessee Tech,Texas,UCLA,UNC Greensboro,USC,Utah,Vanderbilt,Vermont,Washington,West Virginia,WI Green Bay,UC San Diego
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Alabama,NaN,0.954425,0.862467,0.372033,0.541847,0.087864,0.652267,0.342667,0.954425,0.835100,0.884582,0.640815,0.874335,0.753463,0.750500,0.768107,0.954425,0.764072,0.581044,0.550366,0.570316,0.287321,0.455343,0.952046,0.928469,0.641058,0.443561,0.497065,0.734790,0.829342,0.600898,0.856105,0.875358,0.921048,0.409789,0.684127,0.950864,0.724483,0.247309,0.595001,0.252251,0.739176,0.766506,0.942274,0.845440,0.744695,0.952046,0.947817,0.077059,0.921624,0.416816,0.185871,0.952046,0.081202,0.140245,0.952046,0.139300,0.742769,0.374175,0.952477,0.758456,0.367937,0.921361,0.954425
Arkansas St,0.046241,NaN,0.136014,0.046241,0.074816,0.046241,0.061890,0.046241,0.676768,0.138150,0.123514,0.048721,0.141297,0.067108,0.138150,0.080245,0.717240,0.066815,0.079564,0.046241,0.066054,0.046241,0.046672,0.301205,0.151207,0.065794,0.046241,0.046428,0.046241,0.061645,0.046241,0.066586,0.256361,0.259487,0.046241,0.080969,0.159623,0.046241,0.046241,0.046241,0.046241,0.061645,0.082139,0.389568,0.075024,0.118245,0.292961,0.263546,0.046241,0.136279,0.046241,0.048721,0.341281,0.046241,0.046241,0.476526,0.046241,0.065671,0.065774,0.292612,0.080969,0.046241,0.132868,0.714069
Ball St,0.140053,0.859297,NaN,0.127437,0.245238,0.046241,0.236148,0.046241,0.862661,0.413658,0.305752,0.244499,0.363310,0.264219,0.339524,0.270448,0.875467,0.263888,0.220087,0.175675,0.236148,0.129318,0.242248,0.825866,0.559752,0.221183,0.048721,0.203733,0.224457,0.221495,0.117810,0.264219,0.319590,0.316750,0.086836,0.261089,0.669545,0.105485,0.048721,0.130390,0.056356,0.245238,0.274833,0.654809,0.244614,0.141442,0.768335,0.586236,0.046241,0.448056,0.052144,0.136411,0.781152,0.046241,0.046241,0.859962,0.049795,0.257495,0.263305,0.823036,0.263761,0.117262,0.336277,0.874217
Baylor,0.620468,0.954425,0.878050,NaN,0.785477,0.142686,0.653449,0.364951,0.954425,0.876843,0.901825,0.706472,0.876109,0.742643,0.841218,0.789013,0.954425,0.803046,0.530797,0.735331,0.801006,0.643303,0.742140,0.952477,0.952477,0.763242,0.323683,0.647489,0.685385,0.750671,0.449941,0.797942,0.910697,0.841170,0.550215,0.691038,0.952477,0.749827,0.249880,0.704612,0.490489,0.607890,0.600344,0.950395,0.783243,0.766673,0.954425,0.954425,0.125606,0.934422,0.327459,0.373152,0.954425,0.142249,0.084966,0.954425,0.216817,0.798224,0.671117,0.954425,0.761857,0.703876,0.928378,0.954425
California,0.392624,0.931992,0.747924,0.209767,NaN,0.059661,0.277877,0.130213,0.939002,0.738938,0.552610,0.499918,0.721801,0.526839,0.731626,0.704961,0.939002,0.315131,0.369112,0.225528,0.326007,0.253563,0.252194,0.861248,0.748786,0.174561,0.206469,0.241574,0.232640,0.153332,0.211973,0.315919,0.750656,0.646526,0.293978,0.444458,0.751396,0.169515,0.075503,0.218345,0.211883,0.371675,0.535098,0.895003,0.516256,0.476825,0.858663,0.855659,0.059661,0.654836,0.122023,0.249753,0.919149,0.059661,0.108926,0.925169,0.116708,0.265431,0.264312,0.859605,0.452633,0.209198,0.728901,0.939002
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Vermont,0.048721,0.703111,0.175389,0.046241,0.135181,0.046241,0.113782,0.046241,0.720350,0.160107,0.197958,0.056581,0.200

In [64]:
df_matrix.to_csv(f'../data/simulations/womens/matchup_matrix_{season}.csv', index=True)
df_matrix_display.to_csv(f'../data/simulations/womens/matchup_matrix_display_{season}.csv', index=True)

'Done'

'Done'